In [27]:
import pandas as pd
import numpy as np 
from pandas_datareader import data as web


In [28]:
start = "2015-01-01"
end = "2026-01-01"

gdp = web.DataReader("GDP", "fred", start, end)
unrate = web.DataReader("UNRATE", "fred", start, end)
nfci = web.DataReader("NFCI", "fred", start, end)

In [29]:
gdp["gdp_growth"] = gdp["GDP"].pct_change() * 100

In [30]:
macro = gdp[["gdp_growth"]].join(unrate).join(nfci).reset_index()

In [31]:
macro = macro.rename(columns = { "DATE" : "date", "UNRATE" : "unemployment", "NFCI" : "financial_conditions"})
macro.head()

,date,gdp_growth,unemployment,financial_conditions
0,2015-01-01,NaN,5.7,NaN
1,2015-04-01,1.197191,5.4,NaN
2,2015-07-01,0.666540,5.2,NaN
3,2015-10-01,0.182109,5.0,NaN
4,2016-01-01,0.492516,4.8,-0.34411


In [32]:
macro["year"] = macro["date"].dt.year

macro_annual = macro.groupby("year").agg({
        "gdp_growth": "mean",
        "unemployment": "mean",
        "financial_conditions": "mean"
    }).reset_index()
macro_annual.head()

,year,gdp_growth,unemployment,financial_conditions
0,2015,0.681947,5.325,NaN
1,2016,0.875899,4.900,-0.355923
2,2017,1.219298,4.400,NaN
3,2018,1.081797,3.900,NaN
4,2019,1.192293,3.750,NaN


In [34]:
macro_annual["gdp_growth_lag1"] = macro_annual["gdp_growth"].shift(1)
macro_annual["unemployment_lag1"] = macro_annual["unemployment"].shift(1)
macro_annual["financial_conditions_lag1"] = macro_annual["financial_conditions"].shift(1)
macro_final = macro_annual[
    ["year", "gdp_growth_lag1", "unemployment_lag1", "financial_conditions_lag1"]]
macro_final.head(10)

,year,gdp_growth_lag1,unemployment_lag1,financial_conditions_lag1
0,2015,NaN,NaN,NaN
1,2016,0.681947,5.325,NaN
2,2017,0.875899,4.900,-0.355923
3,2018,1.219298,4.400,NaN
4,2019,1.081797,3.900,NaN
5,2020,1.192293,3.750,NaN
6,2021,0.359914,8.875,NaN
7,2022,2.953582,5.600,-0.632345
8,2023,1.916286,3.700,-0.267940
9,2024,1.510789,3.575,NaN


In [35]:
macro_final.to_csv(r"C:\Users\Ozoha\Documents\.ipynb_checkpoints\credit-risk-ews\data\macro_dataset.csv", index = False)